# 📦 DACNTT NeuMF — Data Pipeline Walkthrough + Chạy thực nghiệm (Colab GPU)

Notebook này gộp 3 việc:

1. **Phần 0** — Thiết lập môi trường: TỰ ĐỘNG nhận biết đang chạy trên Colab hay local. Trên Colab: mount Drive, lấy code mới nhất từ GitHub, copy dữ liệu, symlink `outputs/` thẳng vào Drive. Chạy local: bỏ qua toàn bộ, dùng ngay repo hiện có.
2. **Phần 1** — Xem code từng module của pipeline (giải thích + hiển thị source).
3. **Phần 2** — Chạy sống từng bước pipeline trên DataCo thật (load → gộp trùng → k-core → re-index → split → feedback weight → TrainDataset), có biểu đồ minh hoạ.
4. **Phần 3** — (tuỳ chọn, cần GPU/nhiều thời gian) Chạy thực nghiệm đầy đủ và lưu kết quả vào Drive.

**Trước khi chạy trên Colab:** `git push` code mới nhất từ máy bạn trước — notebook lấy code qua `git clone`/`git pull`, không tự đồng bộ thay đổi local.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Đang chạy trên Colab:", IN_COLAB)

## Phần 0.1 — Kiểm tra GPU (chỉ có ý nghĩa trên Colab)

In [ ]:
if IN_COLAB:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    else:
        print("⚠️  Chưa bật GPU — Runtime > Change runtime type > GPU, rồi Restart & chạy lại từ đầu.")
else:
    print("Không chạy trên Colab — bỏ qua bước kiểm tra GPU Colab.")

## Phần 0.2 — Mount Google Drive (chỉ Colab)

Tạo sẵn cấu trúc thư mục `DACNTT/` trên Drive để chứa dữ liệu thô + toàn bộ kết quả thực nghiệm — dùng lại được giữa các phiên Colab.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    DRIVE_ROOT = "/content/drive/MyDrive/DACNTT"
    os.makedirs(f"{DRIVE_ROOT}/data_raw/dataco", exist_ok=True)
    os.makedirs(f"{DRIVE_ROOT}/data_raw/hm", exist_ok=True)
    os.makedirs(f"{DRIVE_ROOT}/data_processed_hm", exist_ok=True)
    os.makedirs(f"{DRIVE_ROOT}/outputs", exist_ok=True)
    print("Thư mục Drive dự án:", DRIVE_ROOT)

## Phần 0.3 — Upload dữ liệu thô lên Drive (CHỈ CẦN LÀM 1 LẦN, bỏ qua nếu đã có)

Mở Google Drive trên trình duyệt, vào đúng các thư mục vừa tạo rồi kéo-thả upload:

| File | Đích trên Drive |
|---|---|
| `DataCoSupplyChainDataset.csv` | `DACNTT/data_raw/dataco/` |
| `transactions_train.csv`, `articles.csv`, `customers.csv` (H&M) | `DACNTT/data_raw/hm/` |
| `transactions_clean.parquet`, `user_id_map.parquet`, `item_id_map.parquet` (nếu máy bạn đã tạo cache H&M — xem `data/processed/hm/` local) | `DACNTT/data_processed_hm/` — khuyến nghị, tiết kiệm ~2 phút xử lý mỗi phiên |

Chạy ô dưới để kiểm tra đã có đủ file.

In [ ]:
if IN_COLAB:
    checks = {
        f"{DRIVE_ROOT}/data_raw/dataco/DataCoSupplyChainDataset.csv": "DataCo CSV",
        f"{DRIVE_ROOT}/data_raw/hm/transactions_train.csv": "H&M transactions",
        f"{DRIVE_ROOT}/data_raw/hm/articles.csv": "H&M articles",
    }
    for path, label in checks.items():
        ok = os.path.exists(path)
        size = f"{os.path.getsize(path)/1024/1024:.1f} MB" if ok else "—"
        print(f"{'✓' if ok else '✗ THIẾU'}  {label:<20} {size}")
    has_hm_cache = os.path.exists(f"{DRIVE_ROOT}/data_processed_hm/transactions_clean.parquet")
    print(f"\nCache H&M trên Drive: {'✓ có sẵn' if has_hm_cache else '✗ chưa có (sẽ tự tạo ở bước sau)'}")

## Phần 0.4 — Lấy code mới nhất từ GitHub (chỉ Colab)

In [ ]:
if IN_COLAB:
    REPO_URL = "https://github.com/sulinh3625/DU-AN-CNTT.git"
    REPO_DIR = "/content/DU-AN-CNTT"
    PROJECT_DIR = f"{REPO_DIR}/neumf_project_v2"

    if os.path.exists(REPO_DIR):
        !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main
    else:
        !git clone {REPO_URL} {REPO_DIR}

    !pip install -q -r {PROJECT_DIR}/requirements.txt
    !cd {PROJECT_DIR} && git log -1 --format='✓ Đang dùng đúng commit mới nhất đã push: %h — %s (%ci)'

## Phần 0.5 — Copy dữ liệu vào đĩa Colab + symlink `outputs/` vào Drive (chỉ Colab)

`outputs/` được **symlink thẳng vào Drive** (không copy sau khi chạy xong) — checkpoint/bảng/biểu đồ ghi ra nằm trên Drive ngay lập tức, an toàn nếu Colab ngắt kết nối giữa chừng.

In [ ]:
if IN_COLAB:
    import shutil
    os.makedirs(f"{PROJECT_DIR}/data/raw/dataco", exist_ok=True)
    os.makedirs(f"{PROJECT_DIR}/data/raw/hm", exist_ok=True)
    os.makedirs(f"{PROJECT_DIR}/data/processed/hm", exist_ok=True)

    print("Copy DataCo...")
    shutil.copy(f"{DRIVE_ROOT}/data_raw/dataco/DataCoSupplyChainDataset.csv", f"{PROJECT_DIR}/data/raw/dataco/DataCoSupplyChainDataset.csv")

    print("Copy H&M raw (có thể mất vài phút do transactions_train.csv ~3,49GB)...")
    for fname in ["transactions_train.csv", "articles.csv", "customers.csv"]:
        src = f"{DRIVE_ROOT}/data_raw/hm/{fname}"
        if os.path.exists(src):
            shutil.copy(src, f"{PROJECT_DIR}/data/raw/hm/{fname}")
            print(f"  ✓ {fname}")

    cache_src = f"{DRIVE_ROOT}/data_processed_hm"
    if os.path.exists(f"{cache_src}/transactions_clean.parquet"):
        print("Copy cache H&M có sẵn từ Drive...")
        for fname in os.listdir(cache_src):
            shutil.copy(f"{cache_src}/{fname}", f"{PROJECT_DIR}/data/processed/hm/{fname}")
        print("  ✓ Đã copy cache.")
    else:
        print("Chưa có cache trên Drive — tạo mới (chạy 1 lần, ~2 phút)...")
        !cd {PROJECT_DIR} && python scripts/00_prepare_hm_cache.py
        print("Lưu cache vừa tạo ngược lại lên Drive...")
        for fname in os.listdir(f"{PROJECT_DIR}/data/processed/hm"):
            if fname != ".gitkeep":
                shutil.copy(f"{PROJECT_DIR}/data/processed/hm/{fname}", f"{cache_src}/{fname}")
        print("  ✓ Đã lưu cache lên Drive.")

    outputs_path = f"{PROJECT_DIR}/outputs"
    if os.path.islink(outputs_path):
        os.unlink(outputs_path)
    elif os.path.exists(outputs_path):
        shutil.rmtree(outputs_path)
    os.symlink(f"{DRIVE_ROOT}/outputs", outputs_path)
    print("\noutputs/ ->", os.path.realpath(outputs_path), "(mọi kết quả ghi ra sẽ nằm trên Drive)")

    %cd {PROJECT_DIR}/notebooks
else:
    print("Chạy local — giữ nguyên thư mục hiện tại, dùng thẳng dữ liệu/outputs cục bộ.")

---
# Phần 1 — Xem Code & Chạy sống Pipeline (DataCo)

Từ đây trở xuống giống hệt notebook walkthrough gốc — chạy được cả trên Colab (sau khi Phần 0 xong) lẫn local.

## ⚙️ Setup — thêm project root vào sys.path

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Exists       = {PROJECT_ROOT.exists()}')

## 🛠️ Helper: hiển thị source code có syntax highlight

In [ ]:
import inspect
from IPython.display import display, Markdown

def show_source(obj, title=None):
    src = inspect.getsource(obj)
    name = title or getattr(obj, '__qualname__', getattr(obj, '__name__', str(obj)))
    md = f'### 📄 `{name}`\n\n```python\n{src}\n```'
    display(Markdown(md))

def show_file(rel_path):
    p = PROJECT_ROOT / rel_path
    src = p.read_text(encoding='utf-8')
    md = f'### 📄 `{rel_path}`\n\n```python\n{src}\n```'
    display(Markdown(md))

print('Helper functions ready: show_source(obj), show_file(rel_path)')

---
# Phần 1 — Xem Code từng Module


## 1.1 DataCoAdapter — Load & chuẩn hoá raw CSV

**Vai trò:** Đọc raw CSV (encoding latin1), rename cột thành chuẩn `user_raw / item_raw / timestamp / value_raw / source_order`. `source_order` là chỉ số dòng gốc — tie-breaker deterministic khi nhiều giao dịch cùng timestamp.

In [ ]:
from src.data_pipeline.adapters.dataco import DataCoAdapter
show_source(DataCoAdapter)

## 1.2 aggregate_unique_user_item — Gộp transaction lặp

**Vai trò:** Gộp nhiều transaction của cùng (user, item) thành một dòng. Giữ `interaction_count`, `value_sum`, `first/last_timestamp`, `last_source_order`.

In [ ]:
from src.data_pipeline.preprocessing import aggregate_unique_user_item
show_source(aggregate_unique_user_item)

## 1.3 iterative_k_core — Lọc dense users/items

**Vai trò:** Lọc bipartite graph đến k-core. Lặp đến fixed point: xoá user có < k item, xoá item có < k user. Đảm bảo mọi node còn lại có đủ dữ liệu.

In [ ]:
from src.data_pipeline.kcore import iterative_k_core
show_source(iterative_k_core)

## 1.4 reindex_interactions — Chuyển sang integer index

**Vai trò:** Map `user_raw` → `user` (int 0..N-1) và tương tự cho item. Mô hình NeuMF dùng embedding lookup nên cần index liên tục.

In [ ]:
from src.data_pipeline.preprocessing import reindex_interactions
show_source(reindex_interactions)

## 1.5 temporal_leave_one_out — Chia tập train/val/test

**Vai trò:** Temporal LOO split — test = item cuối, val = item áp chót, train = còn lại. User có < `min_interactions` chỉ vào train.

In [ ]:
from src.data_pipeline.splitting import temporal_leave_one_out
show_source(temporal_leave_one_out)

## 1.6 apply_feedback_weights — Gán sample weight

**Vai trò:** Gán `sample_weight` cho BCE loss. `binary`: weight=1. `weighted_confidence`: weight = 1 + α × normalize(log1p(value_sum)).

In [ ]:
from src.data_pipeline.preprocessing import apply_feedback_weights
show_source(apply_feedback_weights)

## 1.7 TrainDataset & negative_sampling — Dataset PyTorch

**Vai trò:** `TrainDataset` sinh negative sample động mỗi epoch (`resample()`). Với mỗi positive (user, item), sinh `neg_ratio` negatives từ catalog trừ train positives của user.

In [ ]:
from src.data_pipeline.dataset import TrainDataset
from src.data_pipeline.negative_sampling import sample_train_negatives, build_user_positive_sets

show_source(TrainDataset)
show_source(sample_train_negatives)

---
# Phần 2 — Chạy Từng Bước Xử Lý Data

> **Yêu cầu:** File `data/raw/dataco/DataCoSupplyChainDataset.csv` phải tồn tại.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

RAW_CSV = PROJECT_ROOT / 'data' / 'raw' / 'dataco' / 'DataCoSupplyChainDataset.csv'
if not RAW_CSV.exists():
    print(f'❌ Không tìm thấy: {RAW_CSV}')
    print('   Đặt file CSV vào đúng thư mục rồi chạy lại.')
else:
    size_mb = RAW_CSV.stat().st_size / 1024 / 1024
    print(f'✅ Tìm thấy: {RAW_CSV.name}  ({size_mb:.1f} MB)')

## Bước 1 — Load raw CSV & chuẩn hoá cột

In [ ]:
adapter = DataCoAdapter(RAW_CSV, encoding='latin1')
events = adapter.load_events()

print('=== Bước 1: Raw events sau load ===')
print(f'  Số dòng    : {len(events):,}')
print(f'  Cột        : {list(events.columns)}')
print(f'  Users raw  : {events["user_raw"].nunique():,}')
print(f'  Items raw  : {events["item_raw"].nunique():,}')
print(f'  Thời gian  : {events["timestamp"].min().date()} → {events["timestamp"].max().date()}')
events.head(5)

In [ ]:
print('=== Phân phối value_raw (Sales) ===')
display(events['value_raw'].describe().to_frame().T.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(events['value_raw'].clip(0, events['value_raw'].quantile(0.99)),
             bins=60, color='#4C72B0', edgecolor='white', linewidth=0.5)
axes[0].set_title('Phân phối Sales (raw, clip 99%)', fontsize=12)
axes[0].set_xlabel('Sales'); axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

monthly = events.set_index('timestamp').resample('ME').size()
axes[1].bar(monthly.index, monthly.values, width=25, color='#55A868', edgecolor='white', linewidth=0.5)
axes[1].set_title('Số giao dịch theo tháng', fontsize=12)
axes[1].set_xlabel('Tháng'); axes[1].set_ylabel('Giao dịch')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout(); plt.show()

## Bước 2 — Gộp duplicate (user, item) pairs

In [ ]:
aggregated = aggregate_unique_user_item(events)

n_raw = len(events)
n_agg = len(aggregated)
dup_rate = (n_raw - n_agg) / n_raw * 100

print('=== Bước 2: Sau aggregate ===')
print(f'  Trước  : {n_raw:,} dòng (transaction)')
print(f'  Sau    : {n_agg:,} dòng (unique user-item pairs)')
print(f'  Giảm   : {n_raw - n_agg:,} dòng ({dup_rate:.1f}% duplicate)')
aggregated.head(5)

In [ ]:
cnt = aggregated['interaction_count']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(cnt[cnt <= cnt.quantile(0.99)], bins=40, color='#C44E52', edgecolor='white', linewidth=0.5)
axes[0].set_title('Số lần mua lặp / (user, item) pair', fontsize=12)
axes[0].set_xlabel('interaction_count'); axes[0].set_ylabel('Count')

user_deg = aggregated.groupby('user_raw')['item_raw'].nunique()
axes[1].hist(user_deg[user_deg <= user_deg.quantile(0.99)], bins=50, color='#8172B2', edgecolor='white', linewidth=0.5)
axes[1].set_title('Số items / user (user degree)', fontsize=12)
axes[1].set_xlabel('items'); axes[1].set_ylabel('Users')

plt.tight_layout(); plt.show()
print(f'Median items/user: {user_deg.median():.0f}  |  Max: {user_deg.max()}')

## Bước 3 — K-core filtering (k = 5)

In [ ]:
K_CORE = 5
k_results = []
for k in [1, 2, 3, 5, 10, 15, 20]:
    f = iterative_k_core(aggregated, k)
    u = f['user_raw'].nunique() if len(f) else 0
    i = f['item_raw'].nunique() if len(f) else 0
    n = len(f)
    k_results.append({'k': k, 'users': u, 'items': i, 'interactions': n,
                       'density_%': n / (u * i) * 100 if u and i else 0})

k_df = pd.DataFrame(k_results)
print('=== Bước 3: K-core sensitivity ===')
display(k_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, label, color in zip(axes,
        ['users', 'items', 'interactions'],
        ['Users còn lại', 'Items còn lại', 'Interactions còn lại'],
        ['#4C72B0', '#55A868', '#C44E52']):
    ax.plot(k_df['k'], k_df[metric], 'o-', color=color, linewidth=2, markersize=7)
    ax.axvline(K_CORE, color='gray', linestyle='--', alpha=0.7, label=f'k={K_CORE}')
    ax.set_xlabel('k'); ax.set_title(label, fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.legend(fontsize=8)

plt.suptitle('Tác động của k-core lên kích thước dataset', fontsize=13)
plt.tight_layout(); plt.show()

filtered = iterative_k_core(aggregated, K_CORE)
print(f'✅ k={K_CORE}: {filtered["user_raw"].nunique():,} users, {filtered["item_raw"].nunique():,} items, {len(filtered):,} interactions')

## Bước 4 — Re-index (user_raw/item_raw → int 0..N)

In [ ]:
data = reindex_interactions(filtered)

print('=== Bước 4: Sau re-index ===')
print(f'  n_users      : {data.n_users:,}')
print(f'  n_items      : {data.n_items:,}')
print(f'  interactions : {len(data.df):,}')
print()
data.df[['user_raw','user','item_raw','item','last_timestamp','value_sum']].head(8)

In [ ]:
u_range = data.df['user'].agg(['min','max'])
i_range = data.df['item'].agg(['min','max'])
print(f'user  index: [{u_range["min"]}, {u_range["max"]}]  expected [0, {data.n_users-1}]', '✅' if u_range['max'] == data.n_users-1 else '❌')
print(f'item  index: [{i_range["min"]}, {i_range["max"]}]  expected [0, {data.n_items-1}]', '✅' if i_range['max'] == data.n_items-1 else '❌')
print(f'Unique users: {data.df["user"].nunique()} == n_users {data.n_users}?', '✅' if data.df['user'].nunique() == data.n_users else '❌')
print(f'Unique items: {data.df["item"].nunique()} == n_items {data.n_items}?', '✅' if data.df['item'].nunique() == data.n_items else '❌')

## Bước 5 — Temporal Leave-One-Out split

In [ ]:
from src.data_pipeline.splitting import assert_disjoint_splits

MIN_LOO = 3
train, val, test = temporal_leave_one_out(data.df, min_interactions=MIN_LOO)

print(f'=== Bước 5: Temporal LOO split (min_interactions={MIN_LOO}) ===')
print(f'  Train        : {len(train):,} interactions  | {train["user"].nunique():,} users')
print(f'  Validation   : {len(val):,} interactions  | {val["user"].nunique():,} users')
print(f'  Test         : {len(test):,} interactions  | {test["user"].nunique():,} users')

assert_disjoint_splits(train, val, test)
print()
print('✅ assert_disjoint_splits PASSED — không có overlap giữa splits')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sizes = [len(train), len(val), len(test)]
clrs  = ['#4C72B0', '#55A868', '#C44E52']
lbls  = ['Train', 'Validation', 'Test']
axes[0].pie(sizes, labels=lbls, colors=clrs, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('Tỷ lệ phân chia split', fontsize=12)

for df_s, label, color in zip([train, val, test], lbls, clrs):
    axes[1].hist(df_s['last_timestamp'].values.astype('datetime64[M]').astype(float),
                 bins=30, alpha=0.6, label=label, color=color)
axes[1].set_title('Phân phối timestamp theo split', fontsize=12)
axes[1].set_xlabel('Tháng (encoded)'); axes[1].legend()

plt.tight_layout(); plt.show()

print(f'Train  : {train["last_timestamp"].min().date()} → {train["last_timestamp"].max().date()}')
print(f'Val    : {val["last_timestamp"].min().date()} → {val["last_timestamp"].max().date()}')
print(f'Test   : {test["last_timestamp"].min().date()} → {test["last_timestamp"].max().date()}')

## Bước 6 — Feedback weights

In [ ]:
train_b, val_b, test_b, meta_b = apply_feedback_weights(train, val, test, mode='binary')
train_w, val_w, test_w, meta_w = apply_feedback_weights(train, val, test,
                                                         mode='weighted_confidence', confidence_alpha=1.0)
print('Binary mode sample_weight:', train_b['sample_weight'].describe()[['min','mean','max']].to_dict())
print('Meta binary:', meta_b)
print()
print('Weighted confidence sample_weight:')
display(train_w['sample_weight'].describe().to_frame().T.round(4))
print('Meta weighted:', meta_w)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_b['sample_weight'], bins=10, color='#4C72B0', edgecolor='white')
axes[0].set_title('Binary mode — sample_weight', fontsize=11)
axes[0].set_xlabel('weight'); axes[0].set_ylabel('Count')

axes[1].hist(train_w['sample_weight'], bins=50, color='#C44E52', edgecolor='white', linewidth=0.3)
axes[1].set_title('Weighted confidence — sample_weight (train)', fontsize=11)
axes[1].set_xlabel('weight')

plt.tight_layout(); plt.show()

## Bước 7 — Tạo TrainDataset và kiểm tra batch

In [ ]:
from torch.utils.data import DataLoader
from src.data_pipeline.dataset import TrainDataset
from src.data_pipeline.negative_sampling import build_user_positive_sets

train_positive_sets = build_user_positive_sets(train_b, n_users=data.n_users)

NEG_RATIO = 4
train_dataset = TrainDataset(
    train_df=train_b,
    n_items=data.n_items,
    train_positive_sets=train_positive_sets,
    neg_ratio=NEG_RATIO,
    seed=42,
)

print(f'=== Bước 7: TrainDataset ===')
print(f'  Positives    : {len(train_b):,}')
print(f'  neg_ratio    : {NEG_RATIO}')
print(f'  Dataset size : {len(train_dataset):,} (pos + neg)')
print(f'  Ratio thực   : {len(train_dataset) / len(train_b):.2f}x')

loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
users_b, items_b, labels_b, weights_b = next(iter(loader))
print()
print('Sample batch:')
print(f'  users   : {users_b.tolist()}')
print(f'  items   : {items_b.tolist()}')
print(f'  labels  : {labels_b.tolist()}  (1=pos, 0=neg)')
print(f'  weights : {[round(w, 2) for w in weights_b.tolist()]}')

In [ ]:
print('=== Kiểm tra: negative không trùng train positives ===')
violations = 0
for ub, ib, lb, _ in DataLoader(train_dataset, batch_size=512):
    for u, i, l in zip(ub.numpy(), ib.numpy(), lb.numpy()):
        if l == 0 and int(i) in train_positive_sets[int(u)]:
            violations += 1
if violations == 0:
    print('✅ PASSED — không có negative nào trùng train positives')
else:
    print(f'❌ {violations} violations!')

## Bước 8 — Lưu splits ra disk

In [ ]:
import json

SAVE = True

if SAVE:
    out_dir = PROJECT_ROOT / 'data' / 'splits' / 'dataco'
    out_dir.mkdir(parents=True, exist_ok=True)

    train_b.to_csv(out_dir / 'train.csv', index=False)
    val_b.to_csv(out_dir / 'validation.csv', index=False)
    test_b.to_csv(out_dir / 'test.csv', index=False)

    meta = {'n_users': data.n_users, 'n_items': data.n_items, 'k_core': K_CORE, 'feedback': meta_b}
    (out_dir / 'meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')

    print(f'✅ Đã lưu splits → {out_dir}')
    for f in sorted(out_dir.iterdir()):
        print(f'   {f.name:25s}  {f.stat().st_size/1024:.1f} KB')
else:
    print('SAVE=False, bỏ qua.')

## 📊 Tổng kết pipeline

In [ ]:
rows = [
    {'Bước': '1. Load raw CSV',           'Records': f'{len(events):,}',        'Mô tả': 'Transaction records gốc'},
    {'Bước': '2. Aggregate duplicates',   'Records': f'{len(aggregated):,}',     'Mô tả': 'Unique (user, item) pairs'},
    {'Bước': f'3. K-core (k={K_CORE})',   'Records': f'{len(filtered):,}',       'Mô tả': f'Sau lọc k-core={K_CORE}'},
    {'Bước': '4. Re-index',               'Records': f'{len(data.df):,}',        'Mô tả': f'{data.n_users:,} users × {data.n_items:,} items'},
    {'Bước': '5. Train split',            'Records': f'{len(train_b):,}',        'Mô tả': 'Temporal LOO — train'},
    {'Bước': '5. Val split',              'Records': f'{len(val_b):,}',          'Mô tả': 'Temporal LOO — validation'},
    {'Bước': '5. Test split',             'Records': f'{len(test_b):,}',         'Mô tả': 'Temporal LOO — test'},
    {'Bước': '7. TrainDataset (1 epoch)', 'Records': f'{len(train_dataset):,}',  'Mô tả': f'Pos + neg (ratio={NEG_RATIO})'},
]
summary = pd.DataFrame(rows)
print('=' * 62)
print('  TỔNG KẾT: Raw CSV → Model-Ready Splits')
print('=' * 62)
display(summary)
print('=' * 62)

---
# Phần 3 — Chạy thực nghiệm đầy đủ (khuyến nghị chạy trên Colab GPU)

Mỗi ô độc lập, chạy theo nhu cầu. `PROJECT_ROOT` đã có sẵn từ Phần 1 (cell Setup) nên các lệnh `run.py` gọi đúng dù cwd hiện tại là `notebooks/`.

### 3a. DataCo — pipeline đầy đủ (sweep + train + multi-seed + weighted-feedback)

In [ ]:
!python {PROJECT_ROOT}/run.py report-full --dataset dataco

### 3b. H&M subset (100k dòng) — pipeline đầy đủ

In [ ]:
!python {PROJECT_ROOT}/run.py report-full --dataset hm_subset

### 3c. H&M quy mô ĐẦY ĐỦ (889.062 user, 90.690 sản phẩm)

Chưa từng chạy được trên CPU (mục 5.3 báo cáo) — khả thi trên GPU. `configs/hm.yaml` đã cấu hình Sampled-99 + lấy mẫu âm vector hoá + tắt LightGCN/BPR-MF. Chỉ chạy `train` (không dùng `report-full` — sweep+multi-seed ở quy mô này sẽ rất tốn thời gian kể cả trên GPU).

In [ ]:
!python {PROJECT_ROOT}/run.py train --dataset hm --run-tag hm_full_colab
!python {PROJECT_ROOT}/run.py evaluate --run-tag hm_full_colab

### 3e. Chạy TOÀN BỘ nối tiếp (khuyến nghị khi đã chắc chắn có GPU) — bấm chạy rồi có thể rời máy

Chạy tuần tự cả 3 thực nghiệm trong 1 ô duy nhất: DataCo (report-full) → H&M subset (report-full) → H&M quy mô đầy đủ (train + evaluate). Mỗi lệnh ghi thẳng kết quả vào Drive ngay khi xong (mục Phần 0.5) nên nếu Colab ngắt kết nối giữa chừng, các thực nghiệm ĐÃ hoàn thành trước đó vẫn được giữ nguyên trên Drive — chỉ mất phần đang chạy dở.

In [ ]:
!python {PROJECT_ROOT}/run.py report-full --dataset dataco
!python {PROJECT_ROOT}/run.py report-full --dataset hm_subset
!python {PROJECT_ROOT}/run.py train --dataset hm --run-tag hm_full_colab
!python {PROJECT_ROOT}/run.py evaluate --run-tag hm_full_colab
print("\n✅ Đã chạy xong cả 3 thực nghiệm — kết quả nằm trên Drive tại DACNTT/outputs/")

### 3d. Xác nhận kết quả đã nằm trên Drive (chỉ có ý nghĩa trên Colab)

In [ ]:
if IN_COLAB:
    import subprocess
    print(subprocess.run(["find", f"{DRIVE_ROOT}/outputs/experiments", "-maxdepth", "1", "-type", "d"], capture_output=True, text=True).stdout)
    print("Mở thư mục DACNTT/outputs/ trong Google Drive trên trình duyệt để xem/tải trực tiếp.")
else:
    print(f"Kết quả nằm tại: {PROJECT_ROOT / 'outputs'}")